# 🚦 Traffic Volume Prediction System
### Exploratory Data Analysis & Machine Learning Modeling

**Developer:** Swapna V  
**Role:** ML Engineer  
**Company:** IPEC Solutions  
**Project:** Traffic Volume Prediction System  

---

## 1. Project Overview
This notebook demonstrates the exploratory data analysis, feature engineering, and model training workflow for predicting hourly urban traffic volume based on temporal and weather conditions.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load dataset
df = pd.read_csv('../data/traffic_data.csv', keep_default_na=False)
df['date'] = pd.to_datetime(df['date'])
df.head()

## 2. Feature Engineering
Extracting hour, month, day of week, and weekend indicator features.

In [2]:
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.dayofweek
df['hour'] = df['date'].dt.hour
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

print('Dataset Shape:', df.shape)
df.describe().T

## 3. Exploratory Data Analysis
Analyzing rush hours and weather impact on traffic volume.

In [3]:
hourly_avg = df.groupby('hour')['traffic_volume'].mean()
plt.figure(figsize=(10, 4))
plt.plot(hourly_avg.index, hourly_avg.values, marker='o', color='#2563EB')
plt.title('Average Traffic Volume by Hour of Day')
plt.xlabel('Hour (0-23)')
plt.ylabel('Avg Traffic Volume (Vehicles/Hour)')
plt.grid(True)
plt.show()

## 4. Model Training & Evaluation
Fitting Gradient Boosting and Random Forest Regressor models.

In [4]:
num_features = ['hour', 'month', 'day_of_week', 'is_weekend', 'temperature', 'rain_1h', 'snow_1h', 'clouds_all']
cat_features = ['holiday', 'weather_main', 'weather_description']

X = df[num_features + cat_features]
y = df['traffic_volume']

split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
])

X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

model = GradientBoostingRegressor(n_estimators=100, random_state=42)
model.fit(X_train_prep, y_train)

preds = model.predict(X_test_prep)
print('R2 Score:', r2_score(y_test, preds))
print('MAE:', mean_absolute_error(y_test, preds))
print('RMSE:', np.sqrt(mean_squared_error(y_test, preds)))